<img src="../assets/orchestration_overview.svg" alt="BTP Orchestration 호출 흐름" style="display:block;width:100%;max-width:1200px;margin:0 auto 24px;border-radius:12px;">


<img src="../../4_agent_patterns/assets/a2a_orchestration_architecture_highlight_tools_mcp.png" alt="Agent Harness A2A Orchestrator architecture with the lesson-relevant area highlighted" style="display:block; width:100%; max-width:1400px; margin:18px auto 26px;">


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
<div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘 개념 · 실습 단계</div>
<div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">1. 등록된 Orchestration 선택 (UI)</h2>
</div>
</div>

In [1]:
from pathlib import Path
import importlib
import sys

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / '1_gen_ai_hub' / 'src').is_dir())
sys.path[:0] = [str(project_root / '1_gen_ai_hub' / 'src'), str(project_root / '3_gen_ai_hub_services')]
import src.orchestration.registry_ui as registry_ui
registry_ui = importlib.reload(registry_ui)

orchestration_registry_ui = registry_ui.build_orchestration_registry_ui()
orchestration_registry_ui.launch(inline=True, share=False, show_error=True, server_name='127.0.0.1', width='100%', height=900)

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


<hr style="border: 0; border-top: 2px solid #c7d2e1; margin: 32px 0 14px;">
<div style="margin: 0 0 18px; color: #4b5f78; font-size: .86em; font-weight: 700; letter-spacing: .04em;">🧩 아래부터 코드 베이스 실습</div>

In [2]:
from pathlib import Path
import sys

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / '1_gen_ai_hub' / 'src').is_dir())
sys.path[:0] = [str(project_root / '1_gen_ai_hub' / 'src'), str(project_root / '3_gen_ai_hub_services')]
from src.orchestration.lesson_support.config import load_settings
from src.orchestration.lab_core import get_orchestration_config, input_placeholders, invoke_saved_orchestration_config, list_orchestration_configs
from src.orchestration.str_lab import (
    SYNTHETIC_STR_DATABASE_RESULT, build_str_database_input,
    summarize_str_database_result,
)
settings = load_settings()

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
<div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘 사례 · STR 초안 작성</div>
<div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">DB 조회 결과: 입력 → Orchestration → 담당자 검토</h2>
</div>
</div>

In [3]:
# INPUT: 가명·마스킹된 DB 조회 결과입니다.
str_database_result = SYNTHETIC_STR_DATABASE_RESULT
str_database_input = build_str_database_input(str_database_result)
{
    'case_id': str_database_result['case_id'],
    'source_tables': ['customer_master', 'account_master', 'ledger_transactions'],
    'ledger_transaction_count': len(str_database_result['ledger_transactions']),
    'orchestration_input': str_database_input,
}


{'case_id': 'STR-TRAIN-2026-001',
 'source_tables': ['customer_master', 'account_master', 'ledger_transactions'],
 'ledger_transaction_count': 4,
 'orchestration_input': '{\n  "account_master": [\n    {\n      "account_no_masked": "110-***-0427",\n      "account_status": "ACTIVE",\n      "currency": "KRW",\n      "customer_id": "CUST-0427",\n      "opened_at": "2023-04-18"\n    }\n  ],\n  "case_id": "STR-TRAIN-2026-001",\n  "customer_master": [\n    {\n      "business_type": "온라인 유통업",\n      "customer_id": "CUST-0427",\n      "customer_name_masked": "김*준",\n      "customer_type": "INDIVIDUAL_BUSINESS",\n      "kyc_last_verified_at": "2026-03-02",\n      "onboarded_at": "2023-04-18",\n      "risk_grade": "MEDIUM"\n    }\n  ],\n  "customer_transaction_summary": [\n    {\n      "avg_daily_credit_krw": 1210000,\n      "avg_daily_debit_krw": 1180000,\n      "customer_id": "CUST-0427",\n      "max_single_credit_krw": 2400000,\n      "period_days": 90,\n      "transaction_count": 86\n    }\n

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
<div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘 입력 해석</div>
<div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">1. 원장 JSON에서 검토 지표 확인하기</h2>
</div>
</div>

In [4]:
# INPUT → OUTPUT: 공개 core API가 DB 원장 값을 사실 기반으로 요약합니다.
transaction_summary = summarize_str_database_result(str_database_result)
transaction_summary

{'case_id': 'STR-TRAIN-2026-001',
 'ledger_transaction_count': 4,
 'total_credit_krw': 9800000,
 'total_debit_krw': 9600000,
 'masked_debit_counterparty_count': 3,
 'time_window_minutes': 16.0,
 'timeline': ['2026-09-14T09:12:00+09:00 | CREDIT | 9,800,000 KRW | ***-***-1842',
  '2026-09-14T09:19:00+09:00 | DEBIT | 3,100,000 KRW | ***-***-7711',
  '2026-09-14T09:23:00+09:00 | DEBIT | 3,200,000 KRW | ***-***-5520',
  '2026-09-14T09:28:00+09:00 | DEBIT | 3,300,000 KRW | ***-***-9034']}

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
<div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘 Prompt 구성</div>
<div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">2. DB 조회 결과 JSON을 Config 입력으로 사용하기</h2>
</div>
</div>

In [5]:
# INPUT → OUTPUT: 검증된 DB 조회 결과를 JSON 문자열로 직렬화합니다.
{
    'input_format': 'application/json',
    'serialized_database_result': str_database_input,
    'character_count': len(str_database_input),
}


{'input_format': 'application/json',
 'serialized_database_result': '{\n  "account_master": [\n    {\n      "account_no_masked": "110-***-0427",\n      "account_status": "ACTIVE",\n      "currency": "KRW",\n      "customer_id": "CUST-0427",\n      "opened_at": "2023-04-18"\n    }\n  ],\n  "case_id": "STR-TRAIN-2026-001",\n  "customer_master": [\n    {\n      "business_type": "온라인 유통업",\n      "customer_id": "CUST-0427",\n      "customer_name_masked": "김*준",\n      "customer_type": "INDIVIDUAL_BUSINESS",\n      "kyc_last_verified_at": "2026-03-02",\n      "onboarded_at": "2023-04-18",\n      "risk_grade": "MEDIUM"\n    }\n  ],\n  "customer_transaction_summary": [\n    {\n      "avg_daily_credit_krw": 1210000,\n      "avg_daily_debit_krw": 1180000,\n      "customer_id": "CUST-0427",\n      "max_single_credit_krw": 2400000,\n      "period_days": 90,\n      "transaction_count": 86\n    }\n  ],\n  "data_limitations": [\n    "모든 식별자는 교육용 가명 또는 마스킹 값입니다.",\n    "상대방 실소유자 정보와 거래 목적 증빙은 본 조회 결과

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
<div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘 개념 · 실습 단계</div>
<div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">3. 등록된 Orchestration 목록 조회</h2>
</div>
</div>

In [6]:
registered = list_orchestration_configs(name = 'aml_report_config'
    # scenario='devday', name='assistant', version='1.0.0',  # 필요하면 필터를 사용하세요.
)
registered

{'count': 1,
 'resources': [{'id': 'c2d8e25e-2f0f-46fe-be41-c100f68d7042',
   'name': 'aml_report_config',
   'version': '0.0.1',
   'scenario': 'orchestration',
   'creation_timestamp': '2026-11-19T01:32:48.814000',
   'managed_by': 'imperative',
   'is_version_head': True}]}

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
<div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘 개념 · 실습 단계</div>
<div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">4. 선택한 Config가 받는 입력 확인</h2>
</div>
</div>

In [7]:
if not registered['resources']:
    raise RuntimeError('등록된 Orchestration이 없습니다. 목록 필터와 resource group을 확인하세요.')

selected_id = registered['resources'][0]['id']  # 목록의 다른 id로 바꿔 선택할 수 있습니다.
selected_config = get_orchestration_config(selected_id)
{
    'config_id': selected_id,
    'input_placeholders': input_placeholders(selected_config),
}


{'config_id': 'c2d8e25e-2f0f-46fe-be41-c100f68d7042',
 'input_placeholders': ['input_var1']}

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
<div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘 Config 계약 확인</div>
<div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">5. DB 조회 결과를 Config placeholder에 매핑하기</h2>
</div>
</div>

In [8]:
# INPUT: Registry Config에서 추출한 placeholder 이름 + DB 조회 결과 JSON
str_placeholder_names = input_placeholders(selected_config)
if len(str_placeholder_names) != 1:
    raise RuntimeError(f'이 STR 실습은 입력 placeholder가 정확히 하나인 Config가 필요합니다. 현재 값: {str_placeholder_names}')

# OUTPUT: SDK run()에 전달될 placeholder_values 미리 보기
str_placeholder_values = {str_placeholder_names[0]: str_database_input}
{
    'config_id': selected_id,
    'placeholder_values': str_placeholder_values,
}


{'config_id': 'c2d8e25e-2f0f-46fe-be41-c100f68d7042',
 'placeholder_values': {'input_var1': '{\n  "account_master": [\n    {\n      "account_no_masked": "110-***-0427",\n      "account_status": "ACTIVE",\n      "currency": "KRW",\n      "customer_id": "CUST-0427",\n      "opened_at": "2023-04-18"\n    }\n  ],\n  "case_id": "STR-TRAIN-2026-001",\n  "customer_master": [\n    {\n      "business_type": "온라인 유통업",\n      "customer_id": "CUST-0427",\n      "customer_name_masked": "김*준",\n      "customer_type": "INDIVIDUAL_BUSINESS",\n      "kyc_last_verified_at": "2026-03-02",\n      "onboarded_at": "2023-04-18",\n      "risk_grade": "MEDIUM"\n    }\n  ],\n  "customer_transaction_summary": [\n    {\n      "avg_daily_credit_krw": 1210000,\n      "avg_daily_debit_krw": 1180000,\n      "customer_id": "CUST-0427",\n      "max_single_credit_krw": 2400000,\n      "period_days": 90,\n      "transaction_count": 86\n    }\n  ],\n  "data_limitations": [\n    "모든 식별자는 교육용 가명 또는 마스킹 값입니다.",\n    "상대방 실소

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
<div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘 개념 · 실습 단계</div>
<div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">6. 선택한 Orchestration 호출</h2>
</div>
</div>

In [ ]:
# CALL: 위에서 확인한 key/value를 명시적으로 전달합니다.
str_draft_result = invoke_saved_orchestration_config(
    None, settings, config_id=selected_id, placeholder_values=str_placeholder_values
)

# OUTPUT: 제출용 보고서가 아닌 담당자 검토용 초안입니다.
str_draft_result